In [9]:
import pandas as pd
import numpy as np

import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.labeling import add_returns
from src.features import add_lagged_returns, add_moving_average_features
from src.volatility import add_close_to_close_volatility

df = pd.read_csv("../data/raw/nifty50.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df = add_returns(df)

df = add_lagged_returns(df)
df = add_moving_average_features(df)

df = add_close_to_close_volatility(df)
df["vol_cc_lag_1"] = df["vol_cc"].shift(1)
df = df.dropna().reset_index(drop=True)
df.tail()

,Date,Open,High,Low,Close,Volume,return,ret_lag_1,ret_lag_5,ret_lag_10,ma_ratio_5,ma_ratio_10,ma_ratio_20,vol_cc,vol_cc_lag_1
4495,2026-02-13,25571.150391,25630.349609,25444.300781,25471.099609,453500,-0.013023,-0.005650,0.001985,-0.003865,0.986987,0.991236,0.998418,0.138191,0.130152
4496,2026-02-16,25423.599609,25697.000000,25372.699219,25682.750000,275800,0.008309,-0.013023,0.006757,-0.009172,0.996614,0.997166,1.006737,0.141515,0.138191
4497,2026-02-17,25637.949219,25764.400391,25570.300781,25725.400391,344100,0.001661,0.008309,0.002623,0.025476,0.999897,0.998830,1.008133,0.140728,0.141515
4498,2026-02-18,25752.650391,25828.050781,25645.150391,25819.349609,310200,0.003652,0.001661,0.000721,0.001883,1.004599,1.002309,1.010652,0.130728,0.140728
4499,2026-02-19,25873.349609,25885.300781,25388.750000,25454.349609,0,-0.014137,0.003652,-0.005650,-0.005168,0.993124,0.988863,0.995786,0.141139,0.130728


In [10]:
features = [
    "ret_lag_1",
    "ret_lag_5",
    "ret_lag_10",
    "ma_ratio_5",
    "ma_ratio_10",
    "ma_ratio_20",
    "vol_cc",
    "vol_cc_lag_1"
]

In [11]:
split = int(len(df) * 0.8)

train = df.iloc[:split]
test = df.iloc[split:]

X_train = train[features]
y_train = train["return"]

X_test = test[features]
y_test = test["return"]

# **Base Quantile Model**
**Using Gradient Boosting**

In [12]:
from sklearn.ensemble import GradientBoostingRegressor

models = {}

for q in [0.1, 0.5, 0.9]:

    model = GradientBoostingRegressor(
        loss="quantile",
        alpha=q,
        n_estimators=200,
        max_depth=3
    )

    model.fit(X_train, y_train)

    models[q] = model

In [13]:
q10_baseline = models[0.1].predict(X_test)
q50_baseline = models[0.5].predict(X_test)
q90_baseline = models[0.9].predict(X_test)

In [14]:
results = test.copy()
results["q10_baseline"] = q10_baseline
results["q50_baseline"] = q50_baseline
results["q90_baseline"] = q90_baseline

results.head()

,Date,Open,High,Low,Close,Volume,return,ret_lag_1,ret_lag_5,ret_lag_10,ma_ratio_5,ma_ratio_10,ma_ratio_20,vol_cc,vol_cc_lag_1,q10_baseline,q50_baseline,q90_baseline
3600,2022-06-30,15774.500000,15890.000000,15728.849609,15780.250000,306000,-0.001193,-0.003224,0.009300,-0.021128,0.999245,1.010174,0.994411,0.176167,0.178987,-0.006292,-0.002117,0.005775
3601,2022-07-01,15703.700195,15793.950195,15511.049805,15752.049805,364100,-0.001787,-0.001193,0.009166,-0.004368,0.996793,1.005417,0.995244,0.176191,0.176167,-0.009339,-0.005124,0.002809
3602,2022-07-04,15710.500000,15852.349609,15661.799805,15835.349609,304300,0.005288,-0.001787,0.008459,0.003704,1.002022,1.007613,1.002833,0.178318,0.176191,-0.004452,0.002093,0.008559
3603,2022-07-05,15909.150391,16025.750000,15785.450195,15810.849609,254200,-0.001547,0.005288,0.001146,0.018804,1.000971,1.004954,1.003205,0.176381,0.178318,-0.009115,-0.002320,0.005549
3604,2022-07-06,15818.200195,16011.349609,15800.900391,15989.799805,288400,0.011318,-0.001547,-0.003224,-0.014419,1.009861,1.012618,1.015740,0.182231,0.176381,0.001685,0.008405,0.013993
